## 1. Setup & Configuration
Installs the libraries and defines the settings used everywhere else.

In [15]:
!pip install -q datasets transformers sentencepiece emoji indic-nlp-library scikit-learn

In [16]:
import os, re, random, unicodedata
import pandas as pd

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    print("No HF_TOKEN secret found; continuing without login (downloads may be rate-limited).")
    
SEED = 42
random.seed(SEED)

LANGS = {   # key = short code used by Samanantar and IndicSentiment
    "hi": {"corp": "hin_Deva", "script": (0x0900, 0x097F)},
    "bn": {"corp": "ben_Beng", "script": (0x0980, 0x09FF)},
    "mr": {"corp": "mar_Deva", "script": (0x0900, 0x097F)},
    "as": {"corp": "asm_Beng", "script": (0x0980, 0x09FF)},
}
N_PAIRS_SCAN  = 300_000   # Samanantar pairs to scan per language (Assamese has only ~141K)
CAP_PER_CLASS = 3_000     # max intent examples per class per language
N_CORPUS      = 30_000    # IndicCorp sentences per language

OUT_DIR = "/kaggle/working/data"
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Text Cleaning Module
One shared cleaning function used for all three datasets.

`clean_text` does the following:
1. Normalizes Unicode (NFC) and Indic-script variants with the IndicNLP normalizer
2. Removes URLs and @mentions
3. Turns hashtags into plain words (`#good_news` → `good news`)
4. Collapses stretched letters ("sooooo" → "sooo")
5. Collapses extra whitespace

**Design choices**
- **Emojis are kept** because they carry sentiment. Use `keep_emoji=False` to remove them.
- **Zero-width joiners (ZWJ/ZWNJ) are kept** because they change how some Indic letters are shaped.
- **Hashtags and mentions are matched with `\S+`, not `\w+`**, because Python's `\w` splits Indic words at vowel signs.

`script_ratio` is the fraction of letters in the expected script, used to drop English or mixed-script lines.

In [17]:
import emoji
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

URL     = re.compile(r"https?://\S+|www\.\S+")
MENTION = re.compile(r"@\S+")
HASHTAG = re.compile(r"#(\S+)")
REPEAT  = re.compile(r"(.)\1{3,}")
ZW      = re.compile(r"[\u200b\ufeff]")      # keep ZWJ/ZWNJ, they matter in Indic scripts

_norm = {}
def get_norm(lang):
    if lang not in _norm:
        try:    _norm[lang] = IndicNormalizerFactory().get_normalizer(lang)
        except Exception: _norm[lang] = None   # fall back to NFC only
    return _norm[lang]

def clean_text(text, lang, keep_emoji=True):
    t = unicodedata.normalize("NFC", str(text))
    t = URL.sub(" ", t)
    t = MENTION.sub(" ", t)
    t = HASHTAG.sub(lambda m: m.group(1).replace("_", " "), t)
    t = ZW.sub("", t)
    if not keep_emoji:
        t = emoji.replace_emoji(t, replace=" ")
    t = REPEAT.sub(r"\1\1\1", t)
    n = get_norm(lang)
    if n: t = n.normalize(t)
    return re.sub(r"\s+", " ", t).strip()

def script_ratio(text, rng):
    letters = [c for c in text if c.isalpha()]
    return sum(rng[0] <= ord(c) <= rng[1] for c in letters) / max(len(letters), 1)

def clean_corpus(sents, lang, script, min_w=4, max_w=100):
    seen, out = set(), []
    for s in sents:
        t = clean_text(s, lang)
        if not (min_w <= len(t.split()) <= max_w): continue
        if script_ratio(t, script) < 0.5:          continue
        if t.endswith("...") or t in seen:         continue
        seen.add(t); out.append(t)
    return out

## 3. Sentiment Data (IndicSentiment)
Product reviews labeled positive or negative.

- Every language contains the **same 1,000 reviews translated**, so label counts are identical across languages. This makes cross-lingual comparison fair.
- The data is **binary** (no neutral class). A few rows have no label and are dropped.
- Only a test file exists per language, so we make our own **70/15/15 stratified split** with a fixed seed. Using the same seed everywhere keeps the splits aligned across languages: a review in the Hindi train split is also in the Bengali train split. This matters for zero-shot evaluation.

In [18]:
from huggingface_hub import HfApi, hf_hub_download
print([f for f in HfApi().list_repo_files("ai4bharat/IndicSentiment", repo_type="dataset")
       if f.endswith(".json")])

['data/test/as.json', 'data/test/bd.json', 'data/test/bn.json', 'data/test/gu.json', 'data/test/hi.json', 'data/test/kn.json', 'data/test/ml.json', 'data/test/mr.json', 'data/test/or.json', 'data/test/pa.json', 'data/test/ta.json', 'data/test/te.json', 'data/test/ur.json', 'data/validation/as.json', 'data/validation/bd.json', 'data/validation/bn.json', 'data/validation/gu.json', 'data/validation/hi.json', 'data/validation/kn.json', 'data/validation/ml.json', 'data/validation/mr.json', 'data/validation/or.json', 'data/validation/pa.json', 'data/validation/ta.json', 'data/validation/te.json', 'data/validation/ur.json']


In [19]:
from sklearn.model_selection import train_test_split

def load_sentiment(path):
    local = hf_hub_download("ai4bharat/IndicSentiment", path, repo_type="dataset")
    try:    df = pd.read_json(local, lines=True)
    except ValueError: df = pd.read_json(local)
    df = df[["INDIC REVIEW", "LABEL"]].rename(columns={"INDIC REVIEW": "text", "LABEL": "sentiment"})
    df["sentiment"] = df["sentiment"].astype(str).str.lower()
    return df

def save_splits(df, name, lang, label_col):
    tr, tmp = train_test_split(df, test_size=0.3, stratify=df[label_col], random_state=SEED)
    va, te  = train_test_split(tmp, test_size=0.5, stratify=tmp[label_col], random_state=SEED)
    for split, part in [("train", tr), ("val", va), ("test", te)]:
        part.to_parquet(f"{OUT_DIR}/{name}_{lang}_{split}.parquet", index=False)

for lang in LANGS:
    df = load_sentiment(f"data/test/{lang}.json")      
    df = df[df.sentiment.isin(["positive", "negative"])].reset_index(drop=True)
    df["text"] = df["text"].map(lambda x: clean_text(x, lang))
    print(lang, len(df), df.sentiment.value_counts().to_dict())
    save_splits(df, "sent", lang, "sentiment")

hi 998 {'positive': 506, 'negative': 492}
bn 998 {'positive': 506, 'negative': 492}
mr 998 {'positive': 506, 'negative': 492}
as 998 {'positive': 506, 'negative': 492}


## 4. Intent Data (Samanantar + Annotation Projection)
No Indic dataset exists with Inquiry / Opinion / Factual labels, so we build one.

**Idea:** label the *English* side of each Samanantar pair with simple rules, then copy that label to the Indic translation.

**Pipeline**
1. Stream English–Indic pairs
2. Drop bad pairs (length, length ratio, wrong script, duplicates)
3. Label the English sentence with rules (the labeler can abstain)
4. **Agreement filter:** keep a pair only if the translation contains "?" exactly when the English label is Inquiry
5. Cap each class at 3,000 and save the splits

**Limitations:** the labels are rule-based (silver, not human), and the text is translated news rather than natural social-media text.

### 4.1 Rule-based English labeler
- **Inquiry:** ends with "?"
- **Opinion:** stance phrases ("I think") or evaluative words ("worst", "wonderful")
- **Factual:** a clean declarative sentence with a number or named entity and no first-person pronoun
- anything else is skipped

In [20]:
WH  = ("what","why","how","when","where","who","whom","whose","which")
OPINION_PHRASES = ("i think","i believe","i feel","i guess","i hope","i wish","in my opinion",
                   "in my view","we believe","we think","i don't think","personally")
OPINION_WORDS = {"best","worst","terrible","awful","great","wonderful","amazing","excellent",
                 "horrible","beautiful","disappointing","unfortunate","unfortunately","fantastic",
                 "brilliant","shameful","wrong","love","hate","fortunately"}
FIRST_PERSON = {"i","me","my","we","our","us","mine"}

def english_intent(s):
    s = s.strip()
    if not s: return None
    toks = re.findall(r"[A-Za-z']+", s)
    if s.endswith("?"):
        return "inquiry"
    low = s.lower()
    # lowercase-only match so "Great Britain" doesn't count as opinion
    if any(p in low for p in OPINION_PHRASES) or any(t in OPINION_WORDS for t in toks):
        return "opinion"
    has_first  = any(t.lower() in FIRST_PERSON for t in toks)
    has_entity = bool(re.search(r"\d", s)) or any(t[0].isupper() for t in toks[1:])
    if s.endswith(".") and not has_first and has_entity and s[0] not in "\"'“‘":
        return "factual"
    return None   # abstain

### 4.2 Inspect the dataset
Checks one record to confirm the column names: `src` is English and `tgt` is the Indic translation.

In [21]:
from datasets import load_dataset
peek = next(iter(load_dataset("ai4bharat/samanantar", "hi", split="train", streaming=True)))
print(peek)

{'idx': 0, 'src': "However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles", 'tgt': 'आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।'}


### 4.3 Build the intent dataset
Runs the pipeline above for each language. The printed line shows how many examples survived the filters and the class distribution **before** capping. Factual dominates, which is why the 3,000-per-class cap is applied. The capped data is therefore **artificially balanced**, so report macro F1 rather than accuracy alone.

In [22]:
def build_intent(lang, info):
    ds = load_dataset("ai4bharat/samanantar", lang, split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=50_000).take(N_PAIRS_SCAN)
    rows, seen = [], set()
    for ex in ds:
        en, tgt = ex["src"].strip(), ex["tgt"].strip()
        ew, tw = len(en.split()), len(tgt.split())
        if not (5 <= ew <= 60 and 4 <= tw <= 80):      continue
        if not (0.5 <= tw / ew <= 2.0):                continue   # misaligned pair
        if script_ratio(tgt, info["script"]) < 0.5:    continue
        label = english_intent(en)
        if label is None:                              continue
        if ("?" in tgt) != (label == "inquiry"):       continue   # agreement filter
        if tgt in seen:                                continue
        seen.add(tgt)
        rows.append({"text": clean_text(tgt, lang), "intent": label, "english": en})
    df = pd.DataFrame(rows)
    print(lang, "kept", len(df), df.intent.value_counts().to_dict())
    parts = [g.sample(min(len(g), CAP_PER_CLASS), random_state=SEED) for _, g in df.groupby("intent")]
    return pd.concat(parts, ignore_index=True)

for lang, info in LANGS.items():
    df = build_intent(lang, info)
    save_splits(df, "intent", lang, "intent")

hi kept 126183 {'factual': 115193, 'opinion': 6061, 'inquiry': 4929}
bn kept 84412 {'factual': 69873, 'inquiry': 9450, 'opinion': 5089}
mr kept 102678 {'factual': 90252, 'inquiry': 8267, 'opinion': 4159}
as kept 31807 {'factual': 24789, 'inquiry': 4740, 'opinion': 2278}


### 4.4 Manual audit
Exports 40 test examples per class per language to `intent_audit.csv`.

**TODO:** fill in the `gold` column by hand, then compute precision per class. This gives (a) an estimate of how accurate the rule labels are and (b) a small human-labeled test set for Week 3.

In [23]:
audit = []
for lang in LANGS:
    te = pd.read_parquet(f"{OUT_DIR}/intent_{lang}_test.parquet")
    for _, g in te.groupby("intent"):
        audit.append(g.sample(min(len(g), 40), random_state=SEED).assign(lang=lang, gold=""))
pd.concat(audit).to_csv(f"{OUT_DIR}/intent_audit.csv", index=False)

## 5. IndicCorpV2: Unlabeled Text & Tokenizer Analysis
IndicCorpV2 has no labels. We use it for two things: a pool of clean unlabeled text, and measuring how well each tokenizer handles each language.

### 5.1 Sample and clean
The files are huge, so we read about 20 random chunks per language directly from the Hugging Face files instead of downloading everything. Lines are decoded loosely to skip rare invalid bytes. Each sample then goes through `clean_corpus`, which removes lines that are too short or long, in the wrong script, truncated ("..."), or duplicated.

In [24]:
import fnmatch, time
from huggingface_hub import HfFileSystem
fs = HfFileSystem()

CORP_FILES = {"hin_Deva": "hi-*.txt", "ben_Beng": "bn.txt",
              "mar_Deva": "mr.txt",   "asm_Beng": "as.txt"}

def with_retry(fn, tries=6, wait=20):
    """Re-run fn if Hugging Face says 'too many requests'."""
    for k in range(tries):
        try:
            return fn()
        except Exception as e:
            if "429" in str(e) and k < tries - 1:
                print(f"Rate limited, waiting {wait * (k + 1)}s ...")
                time.sleep(wait * (k + 1))
            else:
                raise

# ONE listing call instead of one per language; it also gives the file sizes
listing = with_retry(lambda: fs.ls("datasets/ai4bharat/IndicCorpV2/data", detail=True))
FILE_SIZES = {e["name"]: e["size"] for e in listing if e["type"] == "file"}

def sample_indiccorp(code, n, n_chunks=20, seed=SEED):
    rng = random.Random(seed)
    files = sorted(p for p in FILE_SIZES
                   if fnmatch.fnmatch(p.split("/")[-1], CORP_FILES[code]))
    per_chunk = n // n_chunks + 1
    out = []
    for _ in range(n_chunks):
        path = rng.choice(files)
        size = FILE_SIZES[path]
        with fs.open(path, "rb") as f:
            f.seek(rng.randrange(0, max(size - 1_000_000, 1)))
            f.readline()                                   # discard the partial first line
            for _ in range(per_chunk):
                line = f.readline()
                if not line:
                    break
                out.append(line.decode("utf-8", errors="ignore").strip())
    out = [s for s in out if s]
    rng.shuffle(out)
    return out[:n]

corp = {}
for lang, info in LANGS.items():
    sents = with_retry(lambda: sample_indiccorp(info["corp"], N_CORPUS))
    corp[lang] = clean_corpus(sents, lang, info["script"])
    pd.DataFrame({"text": corp[lang]}).to_parquet(f"{OUT_DIR}/corpus_{lang}.parquet", index=False)
    print(lang, len(corp[lang]))

hi 12819
bn 12481
mr 12887
as 12917


### 5.2 Tokenizer fertility
**Fertility = subword tokens per word.** A value near 1 means the tokenizer rarely splits words. Higher values mean words are broken into many pieces, which usually makes learning harder, especially for agglutinative languages.

| Language | MuRIL | XLM-R |
|---|---|---|
| Hindi | 1.11 | 1.31 |
| Bengali | 1.23 | 1.79 |
| Marathi | 1.35 | 1.67 |
| Assamese | 1.42 | 2.56 |

**Reading:** MuRIL splits words less in every language, and the gap is widest for Assamese. Fertility is only a proxy, so the final model choice will be confirmed by F1 in Week 2.

In [25]:
from transformers import AutoTokenizer
from indicnlp.tokenize import indic_tokenize

def fertility(tok, sents, lang, n=2000):
    sample = random.sample(sents, min(n, len(sents)))
    words = sum(len(indic_tokenize.trivial_tokenize(s, lang)) for s in sample)
    subs  = sum(len(tok.tokenize(s)) for s in sample)
    return subs / words

rows = []
for name in ["xlm-roberta-base", "google/muril-base-cased"]:
    tok = AutoTokenizer.from_pretrained(name)
    for lang in LANGS:
        rows.append({"model": name, "lang": lang, "fertility": round(fertility(tok, corp[lang], lang), 2)})
fert = pd.DataFrame(rows).pivot(index="lang", columns="model", values="fertility")
fert.to_csv(f"{OUT_DIR}/fertility.csv"); print(fert)

model  google/muril-base-cased  xlm-roberta-base
lang                                            
as                        1.42              2.56
bn                        1.23              1.79
hi                        1.11              1.31
mr                        1.35              1.67


## 6. Datasets & DataLoaders
Combines sentiment and intent data into one training stream.

- Sentiment and intent come from different datasets, so **each row has only one label**. The missing label is set to `-100`, which PyTorch's loss function ignores.
- Tokenization and padding happen **per batch** (dynamic padding) to save GPU memory.
- `max_len=128` caps the sequence length.
- The test batch below prints the tensor shapes and label ids as a sanity check.

> **Heads-up for Week 2:** sentiment is only about 10% of the training rows (about 700 vs about 6,300 intent rows for Hindi), so many batches contain no sentiment labels. Rebalance by oversampling sentiment, downsampling intent, or alternating tasks.

In [26]:
import torch
from torch.utils.data import Dataset, DataLoader

SENT2ID   = {"negative": 0, "neutral": 1, "positive": 2}   # adjust to the labels you printed in Step 3
INTENT2ID = {"inquiry": 0, "opinion": 1, "factual": 2}

def load_split(split, langs):
    parts = []
    for lang in langs:
        parts.append(pd.read_parquet(f"{OUT_DIR}/sent_{lang}_{split}.parquet")[["text", "sentiment"]])
        parts.append(pd.read_parquet(f"{OUT_DIR}/intent_{lang}_{split}.parquet")[["text", "intent"]])
    return pd.concat(parts, ignore_index=True)   # missing label columns become NaN

class NLUDataset(Dataset):
    def __init__(self, df):
        self.text = df["text"].tolist()
        self.sent = [SENT2ID.get(s, -100) if isinstance(s, str) else -100 for s in df["sentiment"]]
        self.intent = [INTENT2ID.get(s, -100) if isinstance(s, str) else -100 for s in df["intent"]]
    def __len__(self): return len(self.text)
    def __getitem__(self, i):
        return {"text": self.text[i], "sentiment": self.sent[i], "intent": self.intent[i]}

def make_collate(tokenizer, max_len=128):
    def collate(batch):
        enc = tokenizer([b["text"] for b in batch], truncation=True, max_length=max_len,
                        padding=True, return_tensors="pt")
        enc["sentiment"] = torch.tensor([b["sentiment"] for b in batch])
        enc["intent"]    = torch.tensor([b["intent"] for b in batch])
        return enc
    return collate

tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
train_df = load_split("train", ["hi"])               # zero-shot setup: train on Hindi only
loader = DataLoader(NLUDataset(train_df), batch_size=32, shuffle=True, collate_fn=make_collate(tok))
batch = next(iter(loader))
print({k: v.shape for k, v in batch.items()})
print("sentiment labels:", batch["sentiment"].tolist()[:8])
print("intent labels:   ", batch["intent"].tolist()[:8])

{'input_ids': torch.Size([32, 70]), 'attention_mask': torch.Size([32, 70]), 'sentiment': torch.Size([32]), 'intent': torch.Size([32])}
sentiment labels: [-100, -100, -100, -100, -100, -100, -100, -100]
intent labels:    [1, 1, 0, 2, 1, 2, 2, 1]


## 7. Week 1 Summary

**Produced** (in `/kaggle/working/data`)
- Sentiment train/val/test splits for hi, bn, mr, as
- Intent train/val/test splits for hi, bn, mr, as
- Cleaned unlabeled corpus per language
- Tokenizer fertility table
- Audit sheet for the intent labels

**Key findings**
- MuRIL tokenizes all four languages more efficiently than XLM-R, most of all Assamese.
- Intent labels come from rules, so their quality must be reported from the manual audit.

**Known limitations**
- Sentiment data is binary (no neutral class)
- Intent classes are balanced by capping, unlike the real distribution
- Intent text is translated news, while sentiment text is product reviews (domain mismatch)

**Next (Week 2):** choose the model, build the shared encoder with two heads, and train the baseline with FP16 and gradient accumulation.